# Customer Satisfaction & Order Experience Analysis for an Indian E-Commerce Platform

Beginner data analysis project in Python (`pandas`, `NumPy`, `matplotlib`, `seaborn`) analyzing 250 online shopping orders.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

## 1. Data Loading & Cleaning

In [ ]:
raw_df = pd.read_csv("../data/raw/online_shopping_data.csv")
print("Raw shape:", raw_df.shape)
print("Duplicates:", raw_df.duplicated().sum())
print("Missing values:\n", raw_df.isnull().sum())

df = raw_df.drop_duplicates().reset_index(drop=True)
df["City"] = df["City"].str.strip().str.title().replace({"Delhi Ncr": "Delhi NCR"})
df.loc[(df["Customer_Age"] < 18) | (df["Customer_Age"] > 100), "Customer_Age"] = np.nan

df["Payment_Mode"] = df["Payment_Mode"].fillna(df["Payment_Mode"].mode()[0])
df["Customer_Age"] = df["Customer_Age"].fillna(round(df["Customer_Age"].median())).astype(int)
df["Order_Value_INR"] = df["Order_Value_INR"].fillna(round(df["Order_Value_INR"].median(), 2)).round(2)
df["Discount_Percent"] = df["Discount_Percent"].fillna(round(df["Discount_Percent"].median(), 1)).round(1)

df = df.sort_values("Order_ID").reset_index(drop=True)
df.to_csv("../data/processed/online_shopping_cleaned_data.csv", index=False)
print("Cleaned shape:", df.shape)
df.head()

## 2. Question 1: What does the outcome (`Customer_Satisfaction_Score`) look like overall?

In [ ]:
num_cols = ["Customer_Age", "Items_Count", "Order_Value_INR", "Discount_Percent", "Delivery_Days", "Customer_Satisfaction_Score"]
display(df[num_cols].describe().T.round(2))

## 3. Question 2: Do `Delivery_Days` and `Discount_Percent` affect the outcome?

In [ ]:
outcome = "Customer_Satisfaction_Score"
print(df[num_cols].corr()[outcome].sort_values(ascending=False).round(3))

## 4. Question 3: Does the outcome differ by category columns?

In [ ]:
for col in ["Membership_Tier", "City", "Product_Category", "Payment_Mode"]:
    print(f"\n--- {col} ---")
    display(df.groupby(col)[outcome].agg(["count", "mean", "median", "std"]).sort_values("mean", ascending=False).round(2))

## 5. Question 4: Which records need attention (`Customer_Satisfaction_Score < 5.0`)?

In [ ]:
at_risk = df[df[outcome] < 5.0].copy()
at_risk.to_csv("../data/processed/orders_needing_attention.csv", index=False)
print(f"At-Risk Orders: {len(at_risk)} / {len(df)} ({len(at_risk)/len(df)*100:.1f}%)")
display(at_risk)

## 6. Visualizations

In [ ]:
# Save charts to ../outputs/charts/
plt.figure(figsize=(7, 4.5))
sns.boxplot(data=df, x="Membership_Tier", y=outcome, order=["VIP", "Plus", "Standard"])
plt.axhline(5.0, color="red", linestyle="--", label="At-Risk (< 5.0)")
plt.title("1. Membership Tier Wise Satisfaction Score")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/charts/1_tier_wise_satisfaction.png")
plt.show()

plt.figure(figsize=(7, 4.5))
sns.histplot(df[outcome], bins=15, kde=True)
plt.axvline(5.0, color="red", linestyle="--", label="At-Risk (< 5.0)")
plt.title("2. Customer Satisfaction Score Distribution")
plt.legend()
plt.tight_layout()
plt.savefig("../outputs/charts/2_satisfaction_distribution.png")
plt.show()

plt.figure(figsize=(7, 4.5))
sns.regplot(data=df, x="Delivery_Days", y=outcome)
plt.title("3. Delivery Days vs Satisfaction Score")
plt.tight_layout()
plt.savefig("../outputs/charts/3_delivery_days_vs_satisfaction.png")
plt.show()

plt.figure(figsize=(8, 4.5))
sns.barplot(data=df, x="Product_Category", y=outcome)
plt.title("4. Product Category Wise Satisfaction Score")
plt.tight_layout()
plt.savefig("../outputs/charts/4_category_wise_performance.png")
plt.show()

plt.figure(figsize=(7.5, 4.5))
city_counts = df["City"].value_counts()
sns.barplot(x=city_counts.index, y=city_counts.values)
plt.title("5. Order Volume Distribution by City")
plt.tight_layout()
plt.savefig("../outputs/charts/5_city_distribution.png")
plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("6. Numeric Features Correlation Heatmap")
plt.tight_layout()
plt.savefig("../outputs/charts/6_correlation_heatmap.png")
plt.show()